# PatchCore — 볼트 외관 이상 탐지 노트북

**모델**: PatchCore (메모리 뱅크 기반 딥러닝 이상 탐지)  
**데이터**: 볼트 이미지 — 정상(OK): 2,701장 / 불량(NG): 14장  
**학습 분할**: 정상 2,500장 (랜덤, seed=42) → train / 정상 201장 + 불량 14장 → test  
**참고 논문**: PatchCore: Towards Total Recall in Industrial Anomaly Detection

In [ ]:
# 셀 1 — 패키지 설치
#!pip install anomalib

In [ ]:
# 셀 2 — 라이브러리 임포트
import os, shutil, random, warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch
from sklearn.metrics import (roc_curve, auc as sklearn_auc,
                              accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay)
warnings.filterwarnings('ignore')
print("✅ 라이브러리 임포트 완료")

In [ ]:
# 셀 3 — 디바이스 확인
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU 사용 가능: {gpu_name}")
    print(f"   VRAM: {vram_gb:.1f} GB")
else:
    print("⚠️  GPU 없음 — CPU로 실행됩니다 (학습 시간이 길어질 수 있습니다)")
print(f"현재 디바이스: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# 셀 4 — 파라미터 설정 (총 31개)
# 이 셀에서 모든 파라미터를 자유롭게 수정할 수 있습니다.

# ============================================================
# [섹션 1] Patchcore() 모델 생성 파라미터 (10개)
# ============================================================
backbone              = "wide_resnet50_2"        # 특징 추출 CNN: "wide_resnet50_2", "resnet18" 등
layers                = ["layer2", "layer3"]     # 특징 추출 레이어 목록
pre_trained           = True                     # ImageNet 사전학습 가중치 사용 여부
coreset_sampling_ratio = 0.1                     # 메모리 뱅크 저장 비율 (0.0~1.0)
num_neighbors         = 9                        # k-NN 최근접 이웃 수
precision             = "float32"                # 연산 정밀도: "float32" 또는 "float16"
pre_processor         = True                     # 전처리기 사용 여부
post_processor        = True                     # 후처리기 사용 여부
evaluator             = True                     # 평가기 사용 여부
visualizer            = True                     # 시각화기 사용 여부

# ============================================================
# [섹션 2] 전처리 설정 파라미터 (3개)
# ============================================================
image_size            = (256, 256)               # 입력 이미지 리사이즈 크기 (H, W)
center_crop_size      = None                     # Center crop 크기 (논문 재현: (224,224), None=미사용)
normalization         = "min_max"                # 점수 정규화: "min_max" 또는 "none"

# ============================================================
# [섹션 3] Threshold 설정 파라미터 (2개)
# ============================================================
threshold_method      = "percentile"             # 판정 방식: "percentile" 또는 "absolute"
threshold_value       = 95.0                     # 판정 기준값 (percentile: 0~100)

# ============================================================
# [섹션 4] 데이터 모듈 설정 파라미터 (7개)
# ============================================================
normal_dir            = "train/good"             # 정상 학습 이미지 경로 (root 기준 상대 경로)
abnormal_dir          = "test/defect"            # 불량 테스트 이미지 경로 (root 기준 상대 경로)
normal_test_dir       = "test/good"              # 정상 테스트 이미지 경로 (root 기준 상대 경로)
train_batch_size      = 32                       # 학습 배치 크기
eval_batch_size       = 32                       # 평가 배치 크기
num_workers           = 8                        # 데이터 로더 워커 수
task                  = "classification"         # "classification" 또는 "segmentation"

# ============================================================
# [섹션 5] Engine / Trainer 설정 파라미터 (4개)
# ============================================================
max_epochs            = 1                        # 학습 에폭 수 (PatchCore: 단일 패스로 충분)
gradient_clip_val     = 0                        # gradient clipping 값 (PatchCore: 역전파 없음)
num_sanity_val_steps  = 0                        # sanity check 스텝 수
devices               = 1                        # 사용 디바이스 수

# ============================================================
# [섹션 6] 평가 설정 파라미터 (4개)
# ============================================================
metrics               = ["AUROC"]                # 평가 지표 목록
evaluator_eval        = True                     # 평가기 사용 여부 (평가 섹션)
threshold_method_eval = "percentile"             # 판정 방식 (평가 섹션)
threshold_value_eval  = 95.0                     # 판정 기준값 (평가 섹션)

# ============================================================
# [섹션 7] 실험 재현성 설정 파라미터 (1개)
# ============================================================
seed                  = 42                       # 랜덤 시드

print(f"✅ 파라미터 설정 완료 (총 31개)")
print(f"   backbone={backbone} | coreset_ratio={coreset_sampling_ratio} | image_size={image_size} | seed={seed}")

In [ ]:
# 셀 5 — MVTec 폴더 구조 변환 (원본 bolt/ 폴더 보존)
random.seed(seed)
np.random.seed(seed)

ok_dir = Path("bolt/OK")
ng_dir = Path("bolt/NG")

mvtec_root  = Path("bolt_mvtec")
train_good  = mvtec_root / "train" / "good"
test_good   = mvtec_root / "test"  / "good"
test_defect = mvtec_root / "test"  / "defect"

for d in [train_good, test_good, test_defect]:
    d.mkdir(parents=True, exist_ok=True)

exts = ["*.jpg", "*.jpeg", "*.png", "*.bmp"]
ok_images = sorted([p for e in exts for p in ok_dir.glob(e)])
print(f"OK 이미지 총 수: {len(ok_images)}장")

random.shuffle(ok_images)
train_imgs   = ok_images[:2500]
test_ok_imgs = ok_images[2500:]

for img in train_imgs:
    dst = train_good / img.name
    if not dst.exists(): shutil.copy2(img, dst)

for img in test_ok_imgs:
    dst = test_good / img.name
    if not dst.exists(): shutil.copy2(img, dst)

ng_images = sorted([p for e in exts for p in ng_dir.glob(e)])
for img in ng_images:
    dst = test_defect / img.name
    if not dst.exists(): shutil.copy2(img, dst)

print(f"  → train/good  : {len(list(train_good.glob('*')))}장")
print(f"  → test/good   : {len(list(test_good.glob('*')))}장")
print(f"  → test/defect : {len(list(test_defect.glob('*')))}장")
print("\n✅ MVTec 폴더 변환 완료 (원본 bolt/ 폴더 보존됨)")

In [ ]:
# 셀 6 — 데이터 모듈 생성
from anomalib.data import Folder

datamodule = Folder(
    name="bolt",
    root=str(mvtec_root),
    normal_dir=normal_dir,
    abnormal_dir=abnormal_dir,
    normal_test_dir=normal_test_dir,
    train_batch_size=train_batch_size,
    eval_batch_size=eval_batch_size,
    num_workers=num_workers,
    task=task,
    image_size=image_size,
    seed=seed,
)
datamodule.setup()
print("✅ 데이터 모듈 생성 완료")

In [ ]:
# 셀 7 — 모델 생성
from anomalib.models import Patchcore

model = Patchcore(
    backbone=backbone,
    layers=layers,
    pre_trained=pre_trained,
    coreset_sampling_ratio=coreset_sampling_ratio,
    num_neighbors=num_neighbors,
)
print(f"✅ PatchCore 모델 생성 완료")
print(f"   backbone={backbone} | coreset_ratio={coreset_sampling_ratio} | num_neighbors={num_neighbors}")

In [ ]:
# 셀 8 — 학습 실행 (단일 패스: 역전파 없이 메모리 뱅크 구축)
from anomalib.engine import Engine

engine = Engine(
    max_epochs=max_epochs,
    gradient_clip_val=gradient_clip_val,
    num_sanity_val_steps=num_sanity_val_steps,
    devices=devices,
    accelerator="auto",
)
print(f"🚀 학습 시작 (max_epochs={max_epochs}, 단일 패스로 메모리 뱅크 구축)")
engine.fit(model=model, datamodule=datamodule)
print("\n✅ 학습 완료")

In [ ]:
# 셀 9 — 예측 실행 및 결과 수집
print("🔍 테스트 세트 예측 중...")
predictions = engine.predict(model=model, datamodule=datamodule)

all_images, all_anomaly_maps = [], []
all_scores, all_gt_labels, all_pred_labels = [], [], []

for batch in predictions:
    get = lambda b, k: getattr(b, k) if hasattr(b, k) else b[k]
    all_images.extend(get(batch, 'image').cpu())
    all_anomaly_maps.extend(get(batch, 'anomaly_map').cpu().numpy())
    all_scores.extend(get(batch, 'pred_score').cpu().numpy())
    all_gt_labels.extend(get(batch, 'gt_label').cpu().numpy())
    all_pred_labels.extend(get(batch, 'pred_label').cpu().numpy())

all_scores      = np.array(all_scores,      dtype=np.float32)
all_gt_labels   = np.array(all_gt_labels,   dtype=np.int32)
all_pred_labels = np.array(all_pred_labels, dtype=np.int32)

print(f"✅ 예측 완료")
print(f"   정상(OK): {(all_gt_labels==0).sum()}장 / 불량(NG): {(all_gt_labels==1).sum()}장")

In [ ]:
# 셀 10 — 평가 지표 출력 (ROC Curve, AUROC, Accuracy, Precision, Recall, F1, Confusion Matrix)
fpr, tpr, _ = roc_curve(all_gt_labels, all_scores)
auroc     = sklearn_auc(fpr, tpr)
accuracy  = accuracy_score(all_gt_labels, all_pred_labels)
precision = precision_score(all_gt_labels, all_pred_labels, zero_division=0)
recall    = recall_score(all_gt_labels, all_pred_labels, zero_division=0)
f1        = f1_score(all_gt_labels, all_pred_labels, zero_division=0)
cm        = confusion_matrix(all_gt_labels, all_pred_labels)

print("=" * 46)
print("  PatchCore 평가 지표 요약")
print("=" * 46)
print(f"  AUROC     : {auroc:.4f}")
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print("=" * 46)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, color='darkorange', lw=2,
             label=f'ROC Curve (AUROC = {auroc:.4f})')
axes[0].plot([0,1],[0,1], color='navy', lw=1, linestyle='--', label='Random Classifier')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curve — PatchCore', fontsize=13)
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])

ConfusionMatrixDisplay(cm, display_labels=["OK (정상)", "NG (불량)"]).plot(
    ax=axes[1], colorbar=True, cmap='Blues')
axes[1].set_title('Confusion Matrix — PatchCore', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# 셀 11 — Anomaly Map 시각화 (정상 6장 + 불량 14장 = 20쌍 / 총 40장 출력)
ok_idx = [i for i, l in enumerate(all_gt_labels) if l == 0]
ng_idx = [i for i, l in enumerate(all_gt_labels) if l == 1]

random.seed(seed)
sel_ok = random.sample(ok_idx, min(6, len(ok_idx)))

vis_idx    = sel_ok + ng_idx
vis_labels = ["정상 (OK)"] * len(sel_ok) + ["불량 (NG)"] * len(ng_idx)

n = len(vis_idx)  # 20
fig, axes = plt.subplots(n, 2, figsize=(10, n * 3.2))
fig.suptitle("PatchCore — 원본 이미지 vs Anomaly Map",
             fontsize=16, fontweight='bold', y=1.002)

for row, (idx, lbl) in enumerate(zip(vis_idx, vis_labels)):
    img_np = all_images[idx].permute(1, 2, 0).numpy()
    img_np = np.clip((img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8), 0, 1)
    amap   = all_anomaly_maps[idx]
    amap2d = amap[0] if amap.ndim == 3 else amap
    score  = float(all_scores[idx])
    pred   = "불량 (NG)" if all_pred_labels[idx] == 1 else "정상 (OK)"
    color  = "red" if "NG" in lbl else "green"

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f"[원본] {lbl}  |  예측: {pred}",
                           color=color, fontsize=9, fontweight='bold')
    axes[row, 0].axis("off")

    im = axes[row, 1].imshow(amap2d, cmap="jet", interpolation='bilinear')
    axes[row, 1].set_title(f"[Anomaly Map]  Score: {score:.4f}",
                           color=color, fontsize=9, fontweight='bold')
    axes[row, 1].axis("off")
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()
print(f"\n✅ 시각화 완료: 정상 {len(sel_ok)}장 + 불량 {len(ng_idx)}장 = {n}쌍 (총 {n*2}장)")